<a href="https://colab.research.google.com/github/sreeharik03261119/Internship-project/blob/main/internship_day_4_streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import getpass
ngrok_key = getpass.getpass('Enter ngrok key')

Enter ngrok key··········


In [ ]:
!pip install -q -U streamlit pyngrok

In [ ]:
%%writefile app.py
import pandas as pd
import streamlit as st
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

st.set_page_config(page_title="Crop Predictor", page_icon="🌾", layout="wide")

# ---------------------------------------------------------
# Custom CSS for a colorful look
# ---------------------------------------------------------
st.markdown("""
    <style>
    .stApp {
        background: linear-gradient(135deg, #e8f5e9 0%, #fff9c4 50%, #e1f5fe 100%);
    }
    .main-title {
        text-align: center;
        font-size: 48px;
        font-weight: 800;
        color: black;
        padding: 10px 0 0 0;
    }
    .subtitle {
        text-align: center;
        font-size: 18px;
        color: black;
        margin-bottom: 25px;
    }
    .accuracy-box {
        background: linear-gradient(90deg, #43a047, #66bb6a);
        color: black;
        padding: 14px;
        border-radius: 12px;
        text-align: center;
        font-size: 20px;
        font-weight: 700;
        box-shadow: 0 4px 10px rgba(0,0,0,0.15);
        margin-bottom: 20px;
    }
    .section-header {
        background: linear-gradient(90deg, #ff9800, #ffc107);
        color: black;
        padding: 10px 18px;
        border-radius: 10px;
        font-size: 20px;
        font-weight: 700;
        margin: 20px 0 15px 0;
    }
    div[data-testid="stForm"] {
        background-color: rgba(255, 255, 255, 0.75);
        padding: 25px;
        border-radius: 18px;
        box-shadow: 0 6px 18px rgba(0,0,0,0.12);
    }
    .stButton>button {
        background: linear-gradient(90deg, #2e7d32, #66bb6a);
        color: black;
        font-weight: 700;
        font-size: 18px;
        border-radius: 12px;
        padding: 10px 0;
        border: none;
        width: 100%;
    }
    .stButton>button:hover {
        background: linear-gradient(90deg, #1b5e20, #43a047);
        color: black;
    }
    div[data-testid="stForm"] label {
        color: red !important;
        font-weight: 600;
    }
    </style>
""", unsafe_allow_html=True)

st.markdown('<div class="main-title">Crop Prediction App</div>', unsafe_allow_html=True)
st.markdown('<div class="subtitle">Powered by a Decision Tree model trained on agricultural supply chain data</div>', unsafe_allow_html=True)

# ---------------------------------------------------------
# Load & train (same pipeline as the notebook)
# ---------------------------------------------------------
with st.spinner("🔄 Training model, please wait..."):
    raw = pd.read_csv('/content/agricultural_supply_chain_20_columns.csv')
    raw = raw.drop(columns=['Record_ID']).dropna()

    df_enc = raw.copy()
    cat_cols = df_enc.select_dtypes(include='object').columns.tolist()

    encoders = {}
    for col in cat_cols:
        enc = LabelEncoder()
        df_enc[col] = enc.fit_transform(df_enc[col].astype(str))
        encoders[col] = enc

    target_encoder = encoders['Crop_Name']

    X_clean = df_enc.drop(columns=['Crop_Name'])
    y_clean = df_enc['Crop_Name']

    cluster_scaler = StandardScaler()
    X_scaled = cluster_scaler.fit_transform(X_clean)

    cluster_model = KMeans(n_clusters=3, random_state=42, n_init=10)
    X_clean = X_clean.copy()
    X_clean['Cluster'] = cluster_model.fit_predict(X_scaled)

    X_train, X_test, y_train, y_test = train_test_split(
        X_clean, y_clean, test_size=0.2, random_state=42
    )

    model = DecisionTreeClassifier(criterion='gini', random_state=42)
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)

st.markdown(f'<div class="accuracy-box">✅ Model Accuracy: {round(score * 100, 2)}%</div>', unsafe_allow_html=True)

# ---------------------------------------------------------
# Input form
# ---------------------------------------------------------
st.markdown('<div class="section-header">Crop Prediction App</div>', unsafe_allow_html=True)

manual_fields = [
    'Month', 'Quarter', 'Continent', 'Country',
    'Climate_Zone', 'Currency', 'Farming_System', 'Crop_Category',
    'Commodity_Type', 'Crop_Growth_Stage',
    'Farm_Size_ha', 'Soil_Type', 'Soil_pH', 'Soil_Moisture_pct', 'Rainfall_mm'
]

input_values = {}

with st.form("prediction_form"):
    col1, col2, col3 = st.columns(3)
    columns = [col1, col2, col3]

    for i, col in enumerate(manual_fields):
        target_col = columns[i % 3]
        with target_col:
            if col in encoders:
                options = list(encoders[col].classes_)
                input_values[col] = st.selectbox(f"🌿 {col.replace('_', ' ')}", options)
            else:
                input_values[col] = st.number_input(f"📏 {col.replace('_', ' ')}", value=0.0, step=0.1)

    submitted = st.form_submit_button("🔮 Predict Crop")

# ---------------------------------------------------------
# Prediction
# ---------------------------------------------------------
if submitted:
    row = {}
    for col in manual_fields:
        if col in encoders:
            row[col] = encoders[col].transform([input_values[col]])[0]
        else:
            row[col] = float(input_values[col])

    row['Date'] = encoders['Date'].transform([raw['Date'].mode().iloc[0]])[0]
    row['Year'] = float(raw['Year'].mean())
    row['Crop_Variety'] = encoders['Crop_Variety'].transform([raw['Crop_Variety'].mode().iloc[0]])[0]

    feature_cols = X_clean.drop(columns=['Cluster']).columns
    base_row = pd.DataFrame([row])[feature_cols]
    row['Cluster'] = int(cluster_model.predict(cluster_scaler.transform(base_row))[0])

    final_input = pd.DataFrame([row])[X_clean.columns]
    prediction = model.predict(final_input)
    predicted_crop = target_encoder.inverse_transform(prediction)[0]

    st.markdown(f"""
        <div style="
            background: linear-gradient(90deg, #ff6f00, #ffca28, #43a047);
            padding: 25px;
            border-radius: 18px;
            text-align: center;
            margin-top: 20px;
            box-shadow: 0 6px 18px rgba(0,0,0,0.2);">
            <h2 style="color: black; margin: 0;">🌱 Predicted Crop</h2>
            <p style="color: black; font-size: 34px; font-weight: 800; margin: 10px 0 0 0;">{predicted_crop}</p>
        </div>
    """, unsafe_allow_html=True)

    st.balloons()

Overwriting app.py


In [ ]:
from pyngrok  import ngrok

port = 8501

ngrok.set_auth_token(ngrok_key)
ngrok.connect(port).public_url

'https://consult-cheek-succulent.ngrok-free.dev'

In [ ]:
!rm -rf logs.txt && streamlit run app.py &>/content/logs.txt